# Custom Markets with Regional Partitioning

This notebook combines **user-defined markets** with **regional partitioning**:
- **Partitioning** scopes the scenario to a region (e.g. flights departing from France) using AeroSCOPE data.
- **Markets** define the traffic segments within that scenario (e.g. `domestic`, `intra_eu`, `intercontinental` instead of the default `short_range` / `medium_range` / `long_range`).

Custom markets are defined in `data/markets.yaml`. The configuration in `data/config.yaml` references this file via `models.markets.markets_data_file`.

> **Mockup** --- this notebook is not yet functional.

## 1. Generate partitioned inputs

`create_partitioning` accepts a `markets_file` argument to calibrate RPK and energy shares per user-defined market.

In [ ]:
from aeromaps.utils.functions import create_partitioning

create_partitioning(
    file="data/aeroscope_france_data.csv",
    markets_file="data/markets.yaml",
    path="data",
)

## 2. Create process and compute

In [ ]:
%matplotlib widget
from aeromaps import create_process

process = create_process(configuration_file="data/config.yaml")

In [ ]:
# Parameters can be overridden using market-templated names
process.parameters.cagr_passenger_domestic_reference_periods = [2030]
process.parameters.cagr_passenger_domestic_reference_periods_values = [1.0, 0.5]
process.parameters.hydrogen_final_market_share_intra_eu = 25.0

# other parameters can be defined as always, without market-templated names

process.parameters.load_factor_end_year = 86.0

In [ ]:
process.compute()
process.write_json()

## 3. Inspect results

Output variables use the custom market ids: `rpk_domestic`, `rpk_intra_eu`, `rpk_intercontinental`, etc.

In [ ]:
process.data["vector_outputs"][["rpk_domestic", "rpk_intra_eu", "rpk_intercontinental", "rpk"]]

In [ ]:
process.plot("air_transport_co2_emissions", save=False)